# Syntheia — cloud LLM server

Serves the **cloud tier** of the two-tier architecture: Llama-3.1-8B behind an
Ollama API, reachable from the laptop over a tunnel.

This notebook is **infrastructure, not a demo**. Start it, copy the URL it
prints, and leave the tab open. Nobody looks at this during a demo — it is the
equivalent of starting a database before showing an app.

```
laptop                                 this notebook (Colab/Kaggle T4)
  src/utils/llm_ollama.py  ──────────▶  ollama serve
  OLLAMA_HOST=<tunnel url>   tunnel      llama3.1:8b
```

The wrapper on the laptop is unchanged — `llm_ollama.py` reads `OLLAMA_HOST`
at call time, so pointing it here is one environment variable. Same code path,
which is what keeps the cloud and local benchmark numbers comparable.

### Before you start

- **Runtime → Change runtime type → T4 GPU.** Without a GPU this will run on
  CPU and be unusably slow.
- Budget **10–15 minutes**: ~1 GB for Ollama, ~4.9 GB for the model.
- Free Colab sessions last ~12 h and disconnect after roughly 90 minutes idle.
  Nothing persists — a new session re-pulls everything.


## 1. Check the GPU

If this errors or shows no GPU, fix the runtime type before going further.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install Ollama

~1 GB. Same install script as the laptop.

In [ ]:
!sudo apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Start the server

Colab has no systemd, so Ollama runs as a background process rather than a
service. `OLLAMA_HOST=0.0.0.0` matters — the default binds to localhost only,
which the tunnel cannot reach.

In [ ]:
import os, subprocess, time, requests

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for attempt in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).ok:
            print("Ollama is up")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama did not start — re-run this cell")

## 4. Pull Llama-3.1-8B

~4.9 GB, the slowest step. This is the model the laptop cannot serve: at 4-bit
it needs ~4.7–4.9 GB, against a 4 GB card already sharing room with RoBERTa.
Here there is 16 GB.

In [ ]:
!ollama pull llama3.1:8b
!ollama list

## 5. Open a tunnel

**Cloudflare** is the default because it needs no account. ngrok is included
below as a fallback if Cloudflare is blocked — it needs a free authtoken from
dashboard.ngrok.com.

The URL changes every time you restart. Never hard-code it; it is
configuration.

In [ ]:
# Cloudflare quick tunnel — no signup
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

import re, subprocess, time

proc = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:11434"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
start = time.time()
while time.time() - start < 60:
    line = proc.stdout.readline()
    if not line:
        continue
    found = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if found:
        public_url = found.group(0)
        break

if not public_url:
    raise RuntimeError("No tunnel URL — try the ngrok cell below")

print("=" * 70)
print("  CLOUD ENDPOINT:", public_url)
print("=" * 70)
print("\nOn your laptop, set this and the pipeline uses the cloud tier:\n")
print(f"  export OLLAMA_HOST={public_url}")
print(f"  export OLLAMA_MODEL=llama3.1:8b")
print(f"  export OLLAMA_NUM_CTX=8192      # cloud has VRAM to spare, unlike the laptop")
print("\nVerify from the laptop before relying on it:\n")
print(f"  curl {public_url}/api/tags")

### ngrok — when Cloudflare is blocked on the laptop's network

Some networks reset TLS to `*.trycloudflare.com` (curl error 35 from the laptop,
while the smoke test below still works here). ngrok domains get through.

Token: sign in at dashboard.ngrok.com → **Your Authtoken**. Store it in Colab
**Secrets** (key icon, left sidebar) as `NGROK_AUTHTOKEN` and switch on notebook
access. Do not paste it into this cell — this notebook is tracked in git.

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = os.environ.get("NGROK_AUTHTOKEN")

if not token:
    print("No NGROK_AUTHTOKEN secret -- skipping ngrok (Cloudflare URL above still applies)")
else:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(11434, "http").public_url
    print("=" * 70)
    print("  CLOUD ENDPOINT:", public_url)
    print("=" * 70)
    print(f"\n  export OLLAMA_HOST={public_url}")
    print(f"  curl {public_url}/api/tags")

## 6. Smoke test

Confirm the model answers *here* before blaming the tunnel from the laptop.

In [ ]:
import json, requests

r = requests.post("http://localhost:11434/api/generate", json={
    "model": "llama3.1:8b",
    "prompt": "Reply with only: ready",
    "stream": False,
    "options": {"num_ctx": 8192},
}, timeout=300).json()

print("response:", r.get("response", "").strip()[:80])
tok, secs = r.get("eval_count", 0), r.get("eval_duration", 1) / 1e9
print(f"speed: {tok} tokens in {secs:.2f}s = {tok/secs if secs else 0:.1f} tok/s")

## 7. Keep the session alive

Free Colab reclaims an idle runtime after roughly 90 minutes. Run this cell
before a demo and leave it going — it pings the server every 5 minutes so the
session counts as active.

Stop it with the interrupt button when you are finished.

In [ ]:
import time, requests
from datetime import datetime

print("Keep-alive running. Interrupt to stop.\n")
while True:
    try:
        ok = requests.get("http://localhost:11434/api/tags", timeout=5).ok
        print(f"  {datetime.now():%H:%M:%S}  {'alive' if ok else 'NOT RESPONDING'}")
    except requests.RequestException as e:
        print(f"  {datetime.now():%H:%M:%S}  error: {type(e).__name__}")
    time.sleep(300)

## Demo-day checklist

1. Open this notebook, set **Runtime → T4 GPU**
2. Run cells 1–6 (~10–15 min — do this *before* the audience arrives)
3. Copy the endpoint URL from cell 5
4. On the laptop: `export OLLAMA_HOST=<url>` and `export OLLAMA_MODEL=llama3.1:8b`
5. Verify: `curl $OLLAMA_HOST/api/tags` — must return JSON
6. Start cell 7 and leave this tab open
7. Run the demo from the laptop as normal

**If the session dies mid-demo**, on the laptop:

```bash
unset OLLAMA_HOST
unset OLLAMA_MODEL
```

The pipeline drops to the local `llama3.2:3b` and keeps working. That fallback
is the reason the local tier exists — rehearse it once so it is muscle memory
rather than improvisation.
